# 10 – Better-Quality Input: Fundamental Data

Price/volume factors top out around IC ≈ 0.01–0.03. **Fundamental** value/quality
factors (earnings yield, book-to-price, ROE-like cash-flow yield) are the largest
source of *orthogonal* alpha — information that price history cannot contain.

This sandbox can only reach GitHub, so we use the open
`datasets/s-and-p-500-companies-financials` snapshot (no API key, no registration).
For a clean point-in-time history, run `scripts/ingest_fundamentals_local.py` on
your own machine (SimFin free tier / yfinance — registration steps are in that file).

> **Look-ahead caveat:** the bundled snapshot is dated ≈ Feb 2018 — the *end* of the
> price panel. Using it across 2013–2018 is look-ahead biased (modest for slow value
> ratios, severe for fast signals). Treat results as illustrative of the machinery.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from core.fundamentals_loader import download_fundamentals, load_fundamentals, fundamental_factors
from core.sp500_loader import download_sp500, get_panel, liquid_universe

download_fundamentals(); download_sp500()
f = load_fundamentals()
print('fundamentals:', f.shape)
f[['earnings_yield','book_to_price','sales_yield','ebitda_yield','dividend_yield','size']].describe().round(3).T

## Information Coefficient of each fundamental factor (2013–2018)

Every value/quality factor has **negative** IC here — this is the real, documented
*"value's lost decade"*: cheap stocks badly underperformed growth from 2013–2018.
The signals are not useless — their *sign* is regime-dependent, which is exactly
what the IC-weighting combiner is built to handle.

In [ ]:
uni = liquid_universe(150)
panel = get_panel(uni)
ff = fundamental_factors(list(panel.columns))
fwd = panel.shift(-21) / panel - 1
dates = panel.index[252:-21:21]

rows = []
for col in ff.columns:
    ics = []
    for d in dates:
        pair = pd.concat([ff[col], fwd.loc[d]], axis=1).dropna()
        if len(pair) > 20:
            ic, _ = spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
            if not np.isnan(ic): ics.append(ic)
    rows.append((col, np.mean(ics), len(ics)))
pd.DataFrame(rows, columns=['factor', 'mean_IC', 'n_dates']).round(4)

## Does adding fundamentals help? Only if combined adaptively

Equal-weighting the (wrong-signed) value factors is a disaster; IC-weighting learns
the sign and turns them into a net contributor.

In [ ]:
from strategies.hedge_fund_strategy import HedgeFundStrategy
from backtesting.portfolio_engine import PortfolioEngine

close = get_panel(liquid_universe(120))
vol   = get_panel(liquid_universe(120), field='volume')
fund  = fundamental_factors(list(close.columns))

def run(use_fund, combo):
    hf = HedgeFundStrategy(volume_panel=vol, alpha_combination=combo,
                           construction='decile', gross_leverage=1.0,
                           fundamentals=fund if use_fund else None)
    m = PortfolioEngine('ME', cost_bps=10).run(close, hf.weights).metrics
    return m['sharpe_ratio'], m['max_drawdown']

out = []
for combo in ['equal', 'ic_weighted']:
    s0, d0 = run(False, combo)
    s1, d1 = run(True, combo)
    out.append((combo, 'price-only', round(s0, 3), f'{d0:.1%}'))
    out.append((combo, '+fundamentals', round(s1, 3), f'{d1:.1%}'))
pd.DataFrame(out, columns=['combine', 'inputs', 'Sharpe', 'maxDD'])

## Getting true point-in-time fundamentals (on your machine)

```bash
# free, recommended — true point-in-time, includes delisted names
pip install simfin
#   1. register: https://app.simfin.com/login/register   (free tier, €0)
#   2. copy API key from the dashboard
export SIMFIN_API_KEY=xxxx
export FUND_SOURCE=simfin
python scripts/ingest_fundamentals_local.py
#   -> data/fundamentals_timeseries.parquet  (MultiIndex date,ticker)
```

Then feed a per-rebalance-date slice into the strategy (no look-ahead):
```python
ts = pd.read_parquet('data/fundamentals_timeseries.parquet')
fund_d = ts.xs(rebalance_date, level='date')   # use merge_asof for gaps
HedgeFundStrategy(volume_panel=vol, fundamentals=fund_d, alpha_combination='ic_weighted')
```

**Budget note (<€10):** there is no reputable sub-€10 point-in-time fundamentals
feed worth buying — the SimFin *free* tier is the correct choice. Paid upgrades
(SimFin Pro+ ~$23/mo, Sharadar SF1 ~$50/mo) are above budget and unnecessary for
research at this scale.